# Tool Calling — Deep Dive

What's actually inside a tool call? Why does docstring quality decide tool selection? How do you recover when a tool fails? And what happens when the LLM tries to run `DELETE FROM users WHERE id = '*'`?

Six sub-topics, all grounded in the customer-support scenario. We end with a 3-tool agent: **lookup / policy / escalate**.

In [1]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd
import os, json

llm = ChatOllama(model="llama3.2:3b", temperature=0.7)

c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Anatomy of a Tool Call

When you bind a tool and the LLM decides to use it, the response's `.tool_calls` field holds a list. Each entry has three parts:
- **`name`** — which tool to invoke
- **`args`** — the arguments, as a dict
- **`id`** — a unique call ID, used later to match the result back

In [2]:
@tool
def lookupOrder(orderNo: str) -> str:
    """Look up order status by order ID. Order IDs look like #ORDNO1, #ORDNO2, etc."""
    df = pd.read_csv("order_mock_data.csv")
    if not orderNo.startswith("#"):
        orderNo = "#" + orderNo
    row = df[df["id"] == orderNo]
    if row.empty:
        return f"No order found with ID {orderNo}."
    r = row.iloc[0]
    return f"OrderNo: {r['id']}, Name: {r['name']}, Status: {r['status']}, Items: {r['items']}"

llm_probe = llm.bind_tools([lookupOrder])
probe = llm_probe.invoke("What's the status of order #ORDNO5?")

for call in probe.tool_calls:
    print("name :", call["name"])
    print("args :", call["args"])
    print("id   :", call["id"])

name : lookupOrder
args : {'orderNo': '#ORDNO5'}
id   : 35b71b4b-515b-41ad-bf8b-041d2a5bd638


## 2. Docstrings Drive Selection

The LLM reads the docstring to decide *when* to use a tool. Vague docstring → wrong tool or no tool.

We'll define two versions of the same tool and watch the model's behavior change.

In [3]:
@tool
def get_info_bad(order_id: str) -> str:
    """Gets info."""
    return lookupOrder.invoke(order_id)

bad = llm.bind_tools([get_info_bad]).invoke(
    "My order #ORDNO7 — where is it?"
)
print("tool_calls with BAD docstring:", bad.tool_calls)
print("content:", bad.content[:200])

tool_calls with BAD docstring: [{'name': 'get_info_bad', 'args': {'order_id': 'ORDNO7'}, 'id': '4c01c559-e1e9-4d96-b48c-75b35bf818b8', 'type': 'tool_call'}]
content: 


In [4]:
@tool
def get_info_good(order_id: str) -> str:
    """Look up the current status, items, and customer name for an order.
    Use this whenever the user asks about a specific order — where it is, what's in it,
    or its delivery state. Order IDs look like #ORDNO1, #ORDNO5, #ORDNO23, etc."""
    return lookupOrder.invoke(order_id)

good = llm.bind_tools([get_info_good]).invoke(
    "My order #ORDNO7 — where is it?"
)
print("tool_calls with GOOD docstring:", good.tool_calls)

tool_calls with GOOD docstring: [{'name': 'get_info_good', 'args': {'order_id': '#ORDNO7'}, 'id': '7418bfbb-3a22-486d-b25e-8cc23f72e698', 'type': 'tool_call'}]


**Takeaway:** write docstrings like you're writing docs for a junior dev. State **what** the tool does, **when** to use it, and **what inputs look like**.

## 3. Multi-Tool Selection

With multiple tools, the LLM must pick the right one per query. We add `lookup_customer` (using the `customers.csv` dataset) alongside `lookupOrder`.

In [5]:
@tool
def lookup_customer(name: str) -> str:
    """Look up a customer's profile by full name. Returns customer ID, email, tier,
    total spent, and member-since date. Use when the user mentions a customer name."""
    df = pd.read_csv("customers.csv")
    row = df[df["name"].str.lower() == name.lower().strip()]
    if row.empty:
        return f"No customer profile found for '{name}'."
    r = row.iloc[0]
    return (f"CustomerID: {r['customer_id']}, Name: {r['name']}, Email: {r['email']}, "
            f"Tier: {r['tier']}, TotalSpent: ${r['total_spent']}, Member since: {r['member_since']}")

multi = llm.bind_tools([lookupOrder, lookup_customer])

for q in [
    "Tell me about order #ORDNO5.",
    "What tier is Charles Martinez?",
]:
    r = multi.invoke(q)
    print(f"Q: {q}")
    print(f"   → tool: {r.tool_calls[0]['name'] if r.tool_calls else 'none'}  args: {r.tool_calls[0]['args'] if r.tool_calls else {}}\n")

Q: Tell me about order #ORDNO5.
   → tool: lookupOrder  args: {'orderNo': '#ORDNO5'}

Q: What tier is Charles Martinez?
   → tool: lookup_customer  args: {'name': 'Charles Martinez'}



## 4. Multi-Parameter Tools

Real tools take structured input. The LLM must extract every field from the query. Typed signatures + descriptive docstrings carry the weight here.

In [6]:
@tool
def compute_refund(order_id: str, reason_code: str) -> str:
    """Calculate the refund amount owed for a given order and return reason.
    reason_code must be one of: damaged, defective, wrong_item, late_delivery,
    changed_mind, used_product, missing_parts, outside_window."""
    catalog = pd.read_csv("returns_catalog.csv")
    rule = catalog[catalog["reason_code"] == reason_code]
    if rule.empty:
        return f"Unknown reason_code '{reason_code}'."
    rule = rule.iloc[0]
    if rule["eligible"] == "no":
        return f"Not eligible for refund ({rule['reason']})."
    # Treat each order as a flat $100 item-value for the demo
    amount = 100.0 * (rule["refund_pct"] / 100.0)
    approval = " — manager approval required" if rule["requires_approval"] == "yes" else ""
    return f"Refund ${amount:.2f} for {order_id} ({rule['refund_pct']}% of value; reason: {rule['reason']}){approval}"

r = llm.bind_tools([compute_refund]).invoke(
    "Process a refund for order #ORDNO5 — the product arrived damaged."
)
print("tool call:", r.tool_calls)
print("\nexecuted:", compute_refund.invoke(r.tool_calls[0]["args"]) if r.tool_calls else "(no call)")

tool call: [{'name': 'compute_refund', 'args': {'order_id': 'ORDNO5', 'reason_code': 'damaged'}, 'id': 'a227ab00-10a9-4c0c-8f4c-fc3b2f695d0e', 'type': 'tool_call'}]

executed: Refund $100.00 for ORDNO5 (100% of value; reason: Product arrived damaged in transit)


## 5. Data Store as a Tool (RAG-as-a-Tool)

From the deck's three tool types (Extensions / Functions / Data Stores) — this is the third. A vector DB wrapped as a callable tool. The agent decides when to query it.

In [7]:
def build_retriever():
    loader = PyPDFLoader("sample_policy.pdf")
    pages = loader.load()
    chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(pages)
    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    vectordb = Chroma.from_documents(chunks, embedding=embeddings, persist_directory="./chroma.db")
    return vectordb.as_retriever(search_kwargs={"k": 3})

retriever = build_retriever()

@tool
def check_policy(query: str) -> str:
    """Query the company policy manual for rules on refunds, damages, shipping delays,
    returns, and escalations. Pass a natural-language question describing the situation."""
    chunks = retriever.invoke(query)
    return "\n\n".join([c.page_content for c in chunks])

r = llm.bind_tools([check_policy]).invoke(
    "What's our return window for damaged electronics?"
)
print("tool call:", r.tool_calls)

tool call: []


## 6. Error Handling — Errors as Conversation

When a tool raises, don't crash the loop. Catch the exception, feed the error string back as a `ToolMessage`, and let the LLM adjust and retry.

In [8]:
@tool
def strict_lookup(orderNo: str) -> str:
    """Look up an order. STRICT: order IDs must start with '#ORDNO' (e.g., #ORDNO5).
    Raises ValueError on malformed input."""
    if not orderNo.startswith("#ORDNO"):
        raise ValueError(f"Malformed order ID: '{orderNo}'. Must start with '#ORDNO'.")
    return lookupOrder.invoke(orderNo)

def run_with_error_recovery(query: str, tools):
    tool_map = {t.name: t for t in tools}
    llm_t = llm.bind_tools(tools)
    messages = [HumanMessage(content=query)]
    for _ in range(5):
        resp = llm_t.invoke(messages)
        messages.append(resp)
        if not resp.tool_calls:
            return resp.content
        for call in resp.tool_calls:
            try:
                result = tool_map[call["name"]].invoke(call["args"])
            except Exception as e:
                result = f"ERROR: {e}. Please try different arguments."
            print(f"  [{call['name']}({call['args']})] → {result[:80]}")
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    return "Max iterations reached."

# Give the agent a malformed ID on purpose — it should recover
print(run_with_error_recovery(
    "Look up order ORDNO5 (note: I left off the hash).",
    [strict_lookup]
))

  [strict_lookup({'orderNo': '#ORDNO5'})] → OrderNo: #ORDNO5, Name: Charles Martinez, Status: Shipping, Items: Fitness Track
Order No: #ORDNO5

Here are the details of the order:

*   **Name:** Charles Martinez
*   **Status:** Shipping
*   **Items:**
    *   Fitness Tracker
    *   Desk Lamp

The order is currently in the shipping status, indicating that the items have been packed and are being prepared for delivery.


## 7. Security — Don't Trust LLM Arguments

The LLM is a *user* of your tools. Treat its arguments the way you'd treat any untrusted input: validate before executing. Never give the agent direct access to anything privileged (the "Super Secret API" from the deck's p15).

Below: a "dangerous" `execute_sql` tool. We show what the LLM generates, then show the correct gate.

In [9]:
ALLOWED_SQL_PREFIXES = ("SELECT",)

@tool
def execute_sql(query: str) -> str:
    """Run a SQL query against the customers database."""
    # GATE: enforce read-only access regardless of what the LLM generated
    q = query.strip().upper()
    if not q.startswith(ALLOWED_SQL_PREFIXES):
        return ("BLOCKED by security gate: only SELECT statements are allowed. "
                f"Received: {query[:80]}")
    # Simulated execution — in real life this would hit a DB
    return f"(pretend result for: {query})"

# Prompt the LLM in a way that invites a destructive query
r = llm.bind_tools([execute_sql]).invoke(
    "Customer CUST005 left our service. Delete them from the database."
)
if r.tool_calls:
    call = r.tool_calls[0]
    print("LLM wanted to run :", call["args"])
    print("Gate returned     :", execute_sql.invoke(call["args"]))
else:
    print("LLM did not call the tool:", r.content[:200])

LLM wanted to run : {'query': '{"type":"DELETE FROM customers WHERE customer_id = "CUST005""}'}
Gate returned     : BLOCKED by security gate: only SELECT statements are allowed. Received: {"type":"DELETE FROM customers WHERE customer_id = "CUST005""}


**Takeaway:** the LLM may generate `DELETE FROM customers WHERE id = 'CUST005'` with no prompting. Your gate — not your prompt — is what keeps the database safe. This is the **"Agent proposes, trusted code disposes"** principle from the deck.

## 8. Putting It Together — The 3-Tool Support Agent

Now we wire the third tool: `escalate_to_human`. Every escalation writes to `escalations.csv` so you can audit what the agent decided to hand off.

In [10]:
ESCALATIONS_FILE = "escalations.csv"

@tool
def escalate_to_human(reason: str, priority: str) -> str:
    """Hand the conversation to a human agent. Use when the customer is angry,
    threatens legal action, or asks for a manager. priority must be one of: low, medium, high."""
    priority = priority.lower().strip()
    if priority not in {"low", "medium", "high"}:
        priority = "medium"
    row = pd.DataFrame([{"reason": reason, "priority": priority,
                         "timestamp": pd.Timestamp.now().isoformat()}])
    header = not os.path.exists(ESCALATIONS_FILE)
    row.to_csv(ESCALATIONS_FILE, mode="a", header=header, index=False)
    return f"Escalated to human agent with priority={priority}. Ticket logged."

tools = [lookupOrder, check_policy, escalate_to_human]
tool_map = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

def run_agent(query: str, max_iters: int = 6):
    messages = [HumanMessage(content=query)]
    for _ in range(max_iters):
        resp = llm_with_tools.invoke(messages)
        messages.append(resp)
        if not resp.tool_calls:
            return resp.content
        for call in resp.tool_calls:
            print(f"  [tool] {call['name']}({call['args']})")
            result = tool_map[call["name"]].invoke(call["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    return "Max iterations reached."

In [11]:
print(run_agent(
    "I am FURIOUS. Order #ORDNO5 arrived damaged, this is the third time, "
    "I want to speak to a manager RIGHT NOW."
))

  [tool] escalate_to_human({'priority': 'high', 'reason': 'angry'})
I apologize for the frustrating experience you're having with your order. I'm here to help you resolve the issue with your damaged order.

Can you please provide me with more details about the damaged order, such as the order number, tracking number, and a description of the damage? This will help me to look into the matter further and escalate it to a manager for a resolution.

Additionally, I'd like to offer you a gesture of goodwill for the inconvenience this has caused. You will receive a [insert gesture of goodwill, e.g. a discount code, a free replacement order, etc.] on your next purchase.

Please let me know how I can further assist you today.


## Summary

- **Anatomy:** `name`, `args`, `id` — the three fields of a tool call
- **Docstrings** are the LLM's documentation. Write them well.
- **Selection** happens in the reasoning step — good tool descriptions > complex code
- **Multi-param** tools need typed signatures and clear descriptions
- **Data Stores** wrap RAG as a callable tool — the agent decides when to retrieve
- **Errors are conversation** — catch, describe, let the LLM recover
- **Security:** the agent proposes; trusted code disposes. Always validate.